## ResNet-50: Transfer Learning - `keras`

All training code called here.

- Training Time Taken
- Test Accuracy

For Keras, the documentation is pretty rich, and gives good insight into how to use the api. This is a good starting reference. [https://keras.io/examples/vision/image_classification_from_scratch/](https://keras.io/examples/vision/image_classification_from_scratch/)

### Image Loading

There is difference in how you load images for linear classifiers as opposed to convoluted networks. I had spent time to understand the difference on loading images for `LinearSVC` and `MLPCLassifier` vs a Convolution Neural Network. For CNN training, it's assumed you have a large dataset, way more than what can be loaded in memory, to manage that, Keras supports options to load and classify data from a directory. Since have a small dataset, we can use it off memory.

#### `numpy`

A good amount of time was spent trying to make sure the data was available in a format which is "trainable" for the model. Reading a couple of chapters on `numpy.shape` and `numpy.reshape` helped with the transformation to prepare the data for training.


### Output Details

Each run writes these files, all with the same `RUN_ID`:

- `results/{RUN_ID}/04-resnet/resnet_cv_{RUN_ID}.csv`: 5-fold cross-validation scores
- `results/{RUN_ID}/04-resnet/resnet_cv_{RUN_ID}_fold{n}.csv`: per-epoch metrics for each fold
- `results/{RUN_ID}/04-resnet/resnet_history_{RESNET50_LEARNING_RATE}_{RUN_ID}.csv`: per-epoch metrics for the final fit
- `results/{RUN_ID}/04-resnet/resnet_final_{RUN_ID}.csv`: fit-test score on the 8,000 test images


In [1]:
import datetime, base64, io, os, errno, math

import keras
import platform, sys

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from PIL import Image

from time import perf_counter
import tensorflow as tf

from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPool2D, AveragePooling2D
from keras.models import Sequential

from sklearn.model_selection import StratifiedKFold

from shared_library import (
    SEED,
    cross_validate_keras,
    fit_test,
    get_run_id,
    load_stl10_binary,
    results_dir,
)

In [2]:
RUN_ID = None

In [3]:
RUN_ID = get_run_id(RUN_ID)
RUN_DIR = results_dir(RUN_ID, "04-resnet")

print(f"Architecture: {platform.machine()}")
print(f"Python Version: {sys.version.split()[0]}")
print(f"TensorFlow Version: {tf.__version__}")

gpus = tf.config.list_physical_devices("GPU")
print("GPUs found:", gpus)

Architecture: arm64
Python Version: 3.11.17
TensorFlow Version: 2.16.2
GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### Loading STL-10 Data


In [4]:
# 5000 images training dataset
# path to the binary train file with image data
TRAIN_DATA_PATH = "./stl-10/train_X.bin"
# path to the binary train file with labels
TRAIN_LABEL_PATH = "./stl-10/train_y.bin"
# 8000 images training dataset
# path to the binary test file with image data
TEST_DATA_PATH = "./stl-10/test_X.bin"
# path to the binary test file with labels
TEST_LABEL_PATH = "./stl-10/test_y.bin"

# path to the binary test file with labels
UNLABELED = "./stl-10/unlabeled_X.bin"

# STL-10 dimensions: 96x96 pixels, 3 channels (RGB)
HEIGHT = 96
WIDTH = 96
CHANNELS = 3

NUM_CLASSES = 10

# size of a single image in bytes
SIZE = HEIGHT * WIDTH * CHANNELS
INPUT_SHAPE = (HEIGHT, WIDTH, CHANNELS)


# Usage
X_train, y_train = load_stl10_binary(TRAIN_DATA_PATH, TRAIN_LABEL_PATH)
X_test, y_test = load_stl10_binary(TEST_DATA_PATH, TEST_LABEL_PATH)
# X_unlabeled = load_stl10_binary(UNLABELED)

print("Train shape:", X_train.shape)  # Output: (5000, 96, 96, 3)

# ResNet-50 ImageNet weights expect 0-255 RGB, converted to BGR and mean-centered (not / 255)
X_train = keras.applications.resnet50.preprocess_input(X_train.astype("float32"))
X_test = keras.applications.resnet50.preprocess_input(X_test.astype("float32"))


# One hot encoding labels, everything is mutable...
y_train = tf.keras.utils.to_categorical(y_train, num_classes=10)
y_test = tf.keras.utils.to_categorical(y_test, num_classes=10)

print(y_train[:1])

Train shape: (5000, 96, 96, 3)
[[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]]


### Training Levers

We can train the model by changing parameters listed below


In [5]:
RESNET50_POOLING_AVERAGE = "avg"
DENSE_LAYER_ACTIVATION = "softmax"
OBJECTIVE_FUNCTION = "categorical_crossentropy"

BATCH_SIZE = 64
RESNET50_LEARNING_RATE = 2e-5

RESNET_EPOCHS = 10
CV_FOLDS = 5

FACTOR = 2
LAYERS = NUM_CLASSES * FACTOR

# Common accuracy metric for all outputs, but can use different metrics for different output
LOSS_METRICS = ["accuracy"]

N_JOBS = 5  # parallel processing of jobs

### ResNet-50: Transfer Learning

A transfer learning model is a machine learning technique where a model developed for a source task is reused as the starting point for a different, related target task


In [6]:
def build_resnet(lr):
    input_tensor = keras.Input((224, 224, 3))
    up_sample = (224, 224)

    res50 = keras.applications.ResNet50(
        include_top=False, weights="imagenet", input_tensor=(input_tensor)
    )

    for layer in res50.layers[:143]:
        layer.trainable = False

    model = Sequential(
        [
            keras.layers.Lambda(lambda image: tf.image.resize(image, up_sample)),
            res50,
            keras.layers.Flatten(),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(256, activation="relu"),
            keras.layers.Dropout(0.5),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(128, activation="relu"),
            keras.layers.Dropout(0.5),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dropout(0.5),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(10, activation="softmax"),
        ]
    )

    model.compile(
        loss=keras.losses.CategoricalCrossentropy(),
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        metrics=[keras.metrics.CategoricalAccuracy()],
    )
    return model


def resNet50():
    output = f"{RUN_DIR}/resnet_history_{RESNET50_LEARNING_RATE}_{RUN_ID}.csv"

    model = build_resnet(RESNET50_LEARNING_RATE)

    callbacks = [
        keras.callbacks.CSVLogger(
            filename=output,
            separator=",",
        )
    ]

    # validation comes from the training images, the test images are only used by fit_test
    history = model.fit(
        X_train,
        y_train,
        batch_size=BATCH_SIZE,
        epochs=RESNET_EPOCHS,
        shuffle=True,
        validation_split=0.15,
        callbacks=callbacks,
        verbose=2,
    )
    model.summary()

    print(history)

    return model

#### 5-fold Cross Validation


In [7]:
resnet_cv = cross_validate_keras(
    lambda: build_resnet(RESNET50_LEARNING_RATE),
    "resnet",
    X_train,
    y_train,
    epochs=RESNET_EPOCHS,
    batch_size=BATCH_SIZE,
    run_id=RUN_ID,
    output_dir=RUN_DIR,
    n_splits=CV_FOLDS,
)

2026-11-03 16:13:33.732625: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-11-03 16:13:33.732670: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-11-03 16:13:33.732676: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
2026-11-03 16:13:33.732800: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-11-03 16:13:33.732814: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/10


2026-11-03 16:13:36.665179: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


63/63 - 60s - 956ms/step - categorical_accuracy: 0.2680 - loss: 2.2230 - val_categorical_accuracy: 0.8090 - val_loss: 0.5832
Epoch 2/10
63/63 - 54s - 850ms/step - categorical_accuracy: 0.5125 - loss: 1.4387 - val_categorical_accuracy: 0.8750 - val_loss: 0.4936
Epoch 3/10
63/63 - 67s - 1s/step - categorical_accuracy: 0.6003 - loss: 1.2095 - val_categorical_accuracy: 0.9030 - val_loss: 0.4838
Epoch 4/10
63/63 - 70s - 1s/step - categorical_accuracy: 0.6655 - loss: 1.0540 - val_categorical_accuracy: 0.9280 - val_loss: 0.4724
Epoch 5/10
63/63 - 69s - 1s/step - categorical_accuracy: 0.7145 - loss: 0.9456 - val_categorical_accuracy: 0.9390 - val_loss: 0.4537
Epoch 6/10
63/63 - 72s - 1s/step - categorical_accuracy: 0.7565 - loss: 0.8407 - val_categorical_accuracy: 0.9540 - val_loss: 0.4381
Epoch 7/10
63/63 - 72s - 1s/step - categorical_accuracy: 0.7742 - loss: 0.8024 - val_categorical_accuracy: 0.9580 - val_loss: 0.4092
Epoch 8/10
63/63 - 73s - 1s/step - categorical_accuracy: 0.7962 - loss: 0.

,fold,val_accuracy,val_loss,training_minutes
0,1,0.975,0.356819,11.349525
1,2,0.950,0.389168,12.529940
2,3,0.949,0.406170,11.537813
3,4,0.945,0.455713,12.690053
4,5,0.952,0.421635,12.900555


resnet: 0.9542 +/- 0.0119


#### Fit & Accuracy Test


In [8]:
start_time = perf_counter()
model = resNet50()
fit_test(
    model,
    "resnet",
    X_test,
    y_test,
    training_minutes=(perf_counter() - start_time) / 60,
    batch_size=BATCH_SIZE,
    run_id=RUN_ID,
    output_dir=RUN_DIR,
)

Epoch 1/10
67/67 - 96s - 1s/step - categorical_accuracy: 0.2774 - loss: 2.1788 - val_categorical_accuracy: 0.8253 - val_loss: 0.6000
Epoch 2/10
67/67 - 79s - 1s/step - categorical_accuracy: 0.5219 - loss: 1.4141 - val_categorical_accuracy: 0.8907 - val_loss: 0.5258
Epoch 3/10
67/67 - 80s - 1s/step - categorical_accuracy: 0.6341 - loss: 1.1463 - val_categorical_accuracy: 0.9173 - val_loss: 0.5090
Epoch 4/10
67/67 - 85s - 1s/step - categorical_accuracy: 0.6953 - loss: 1.0158 - val_categorical_accuracy: 0.9240 - val_loss: 0.4946
Epoch 5/10
67/67 - 81s - 1s/step - categorical_accuracy: 0.7365 - loss: 0.9203 - val_categorical_accuracy: 0.9347 - val_loss: 0.4762
Epoch 6/10
67/67 - 82s - 1s/step - categorical_accuracy: 0.7652 - loss: 0.8409 - val_categorical_accuracy: 0.9333 - val_loss: 0.4602
Epoch 7/10
67/67 - 82s - 1s/step - categorical_accuracy: 0.7969 - loss: 0.7717 - val_categorical_accuracy: 0.9413 - val_loss: 0.4367
Epoch 8/10
67/67 - 82s - 1s/step - categorical_accuracy: 0.8221 - los

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 100352)         │       401,408 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 131,542,624 (501.80 MB)

 Trainable params: 40,909,770 (156.06 MB)

 Non-trainable params: 8,813,312 (33.62 MB)

 Optimizer params: 81,819,542 (312.12 MB)

125/125 ━━━━━━━━━━━━━━━━━━━━ 88s 699ms/step - categorical_accuracy: 0.9513 - loss: 0.3804
resnet: test accuracy 0.9513, test loss 0.3804
